# Structured Output using TypeDict

**TypeDict**
1. not very strict
2. returns python dict

In [3]:
from langchain_mistralai import ChatMistralAI 
from typing import TypedDict,Annotated
from dotenv import load_dotenv

In [9]:
class Formated_output(TypedDict):
    CarName : Annotated[str,"Name of the car"]
    CarPrice : Annotated[float,"On road price of car in rs"]
    Summary : Annotated[str,"a breif summary of the car"]

In [4]:
if load_dotenv():
    print("api key successfully loaded")

model = ChatMistralAI(
    model = "mistral-medium-2508",
    temperature = 0.5,
    # max_tokens = 500
)
print("model built successful")

api key successfully loaded
model built successful


Behind the scene of with_structured_output, langchain translates the user requested format into particular model understandable format and that forces the model to generate output in user-defined format, once the output is generated *StructuredOutputParser* of langchain verifies the output and if it difference it handles it.

In [10]:
structured_model = model.with_structured_output(Formated_output)
response = structured_model.invoke("Rolls Royces phantom")
response

{'CarName': 'Rolls-Royce Phantom',
 'CarPrice': 95000000,
 'Summary': "The Rolls-Royce Phantom is the epitome of luxury and sophistication in the automotive world. Known for its iconic design, handcrafted interiors, and unparalleled comfort, the Phantom is a flagship sedan from Rolls-Royce. It features a powerful 6.75-liter V12 engine, delivering a smooth and effortless driving experience. The interior is adorned with the finest materials, including premium leather, wood veneers, and bespoke customization options. Advanced technology, such as the Spirit of Ecstasy Rotating Controller and a state-of-the-art infotainment system, ensures a seamless blend of tradition and innovation. The Phantom is not just a car; it's a statement of prestige and exclusivity."}

# Structured output using Pydantic 
**Pydantic**
1. Strict but handles minor typecasting ie int -> float
2. also returns Py dict 
3. reliable compare to PyDict

In [13]:
from pydantic import BaseModel,Field 
from typing import Optional   

In [32]:
class info_formater(BaseModel):

    CarName : str = Field(description="Name of the Car") 
    CarPrice : float = Field(description="On road price of the given car in rs.")
    Summary : str = Field(description="generate a brief summary.")
    model : Optional[str] = Field(default=None,description="find and retrun specify model name of the car")
    Colors : Optional[list[str]] = Field(default=None,description="find top 5 colors car is available in.")

In [33]:
structured_model = model.with_structured_output(info_formater)
response = structured_model.invoke("""The Rolls-Royce Phantom is the pinnacle of ultra-luxury motoring, serving as the brand's flagship sedan for over a century. It features a 6.75-liter twin-turbo V12 engine, known for a near-silent "magic carpet ride". It offers unparalleled, bespoke customization, advanced technology, and immense road presence, starting around $550,000""")
response

info_formater(CarName='Rolls-Royce Phantom', CarPrice=45500000.0, Summary="The Rolls-Royce Phantom is the epitome of ultra-luxury, representing the brand's flagship sedan for over a century. Powered by a 6.75-liter twin-turbo V12 engine, it delivers a near-silent and smooth 'magic carpet ride.' Known for its unparalleled bespoke customization options, cutting-edge technology, and commanding road presence, the Phantom starts at approximately $550,000.", model=None, Colors=None)

# Structured output with OutputParsers

While using external Output-Parsers always handle *format instructions* always append them to prompt\
**Workflow create parser -> format instruction-> prompt.inovke -> model**

In [47]:
from langchain_core.output_parsers import StrOutputParser # string output parser returns only the string content of a response 
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import HumanMessage,SystemMessage

In [40]:
parser = StrOutputParser()

In [51]:
template = ChatPromptTemplate(
    [
        ("system","You are an {domain} expert. help user with any question regarding the {domain}, if users asks something outside the domain just say cannot answer the Question."),
        ("user","Who is Rohit sharma")
    ]
)

prompt = template.invoke({
    "domain" : "cricket"
})

prompt

ChatPromptValue(messages=[SystemMessage(content='You are an cricket expert. help user with any question regarding the cricket, if users asks something outside the domain just say cannot answer the Question.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Who is Rohit sharma', additional_kwargs={}, response_metadata={})])

In [52]:
response = model.invoke(prompt)
res = parser.invoke(response)
print(res)

Rohit Gurunath Sharma (born **30 April 1987**) is an **Indian international cricketer** and the **current captain of the Indian cricket team** in all formats (Tests, ODIs, and T20Is). He is widely regarded as one of the **greatest opening batsmen** in limited-overs cricket and a **legendary white-ball player**.

### **Key Highlights of Rohit Sharma’s Career:**
#### **1. Batting Records (ODIs & T20Is)**
- **Only player with 3 double centuries in ODIs** (264, 209, 208*).
- **Highest individual ODI score (264 vs Sri Lanka, 2014)** – a world record.
- **Most ODI centuries (31) as an opener** (as of 2024).
- **First player to score 5 T20I centuries** (including **4 in a single year, 2018**).
- **Most T20I runs (4,231+) and centuries (5)** (as of 2024).
- **Only player with 150+ scores in all three formats (Test, ODI, T20I)**.

#### **2. Captaincy (Indian Team & IPL)**
- **Led India to:**
  - **2023 ICC World Test Championship (WTC) Final** (runners-up).
  - **2023 ODI World Cup Final** (run

# Structured output using StructuredOutputParser

In [61]:
from langchain_classic.output_parsers import StructuredOutputParser, ResponseSchema
from langchain_core.prompts import PromptTemplate 

In [76]:
MySchema = [
    ResponseSchema(name="Author",description="Name of Author"),
    ResponseSchema(name="Title",description="Title of poem"),
    ResponseSchema(name="poem",description="poem itself")
]

In [81]:
parser = StructuredOutputParser.from_response_schemas(MySchema)
format_instruction = parser.get_format_instructions()
parser

StructuredOutputParser(response_schemas=[ResponseSchema(name='Author', description='Name of Author', type='string'), ResponseSchema(name='Title', description='Title of poem', type='string'), ResponseSchema(name='poem', description='poem itself', type='string')])

In [ ]:
template = PromptTemplate.from_template(
    "write a haiku on {topic}. and name a fictional character and his age as author \n{format_instruction}"
)

prompt = template.invoke(
    {
        "topic":"cricket",
        "format_instruction":format_instruction
    } 
)
prompt

StringPromptValue(text='write a haiku on cricket. and name a fictional character and his age as author \nThe output should be a markdown code snippet formatted in the following schema, including the leading and trailing "```json" and "```":\n\n```json\n{\n\t"Author": string  // Name of Author\n\t"Title": string  // Title of poem\n\t"poem": string  // poem itself\n}\n```')

In [90]:
response = model.invoke(prompt)

In [94]:
# print(response.content)
res = parser.parse(response.content)
res

{'Author': 'Eldrin Willowbrook (Age: 12)',
 'Title': 'Whispers of Willow',
 'poem': 'Willow cracks—sharp sound,\n\t\tleather kisses summer air,\n\t\tcheers drown the sunset.'}

# Structured Output using PydanticOutputParser

In [6]:
from langchain_core.prompts import PromptTemplate 
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel,Field

In [12]:
class Formater(BaseModel):
    CarName : str = Field(description="Name of the car")
    Price : float = Field(description="price of the car in rupees")
    MaxSpeed : int = Field(description="max speed this car can run")
    Summary : str = Field(description="a very small description about the car.")

parser = PydanticOutputParser(pydantic_object = Formater)
format_instruction = parser.get_format_instructions()


In [19]:
template = PromptTemplate.from_template(
    "you are a dealership agent, for the following car {car_name} return information such as on road price($), max speed(km/s), name, and a small description  \n {format_instruction}"
)

prompt = template.invoke(
    {
        "car_name":"Porche 911 gt3",
        "format_instruction":format_instruction,
    }
)

prompt

StringPromptValue(text='you are a dealership agent, for the following car Porche 911 gt3 return information such as on road price($), max speed(km/s), name, and a small description  \n The output should be formatted as a JSON instance that conforms to the JSON schema below.\n\nAs an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}\nthe object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.\n\nHere is the output schema:\n```\n{"properties": {"CarName": {"description": "Name of the car", "title": "Carname", "type": "string"}, "Price": {"description": "price of the car in rupees", "title": "Price", "type": "number"}, "MaxSpeed": {"description": "max speed this car can run", "title": "Maxspeed", "type": "integer"}, "Summary": {"description": "a very small description about the ca

In [20]:
response = model.invoke(prompt)

In [21]:
print(response.content)

```json
{
  "CarName": "Porsche 911 GT3 (992)",
  "Price": 211300.00,
  "MaxSpeed": 320,
  "Summary": "The Porsche 911 GT3 (992) is a high-performance, track-focused sports car powered by a naturally aspirated 4.0L flat-six engine producing 503 hp. It features rear-wheel drive, a PDK dual-clutch transmission (or optional 6-speed manual), and advanced aerodynamics for unparalleled handling and precision. Designed for purists, it blends raw driving excitement with cutting-edge technology."
}
```
